In [ ]:
#@title Import packages
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
from matplotlib.ticker import MaxNLocator
from matplotlib import gridspec
%matplotlib inline
%config InlineBackend.figure_format = 'svg'
#plt.style.use('seaborn')
import seaborn as sns
import os
import glob
import datetime
import numpy as np
import pandas as pd
import math
from random import randrange
from random import randint
from tqdm import tqdm
import pickle
import re
import json
import pickle


# Functions and settings


In [ ]:
#@title Helper functions
## Def phensys converter for lick rate

#for saving dictionaries
def  savedicti (dictio, name):
            dict_items_save=open("/Users/friedrichjohenning/Desktop/dictionariesneurocorr/"+name+".pickle", "wb")
            pickle.dump(dictio, dict_items_save)

def timestampconvert(x):
  stamp = datetime.timedelta(days = x)
  result = datetime.datetime(1899,12,30,0,0) + stamp
  # print(result.strftime('%Y-%m-%d %H:%M:%S.%f'))
  return result

def datetime_convert_phenosys(csv_path):
    df_pheno = pd.read_csv(csv_path,sep=";",encoding="utf-16")
    #display(list(df_pheno.columns.values))
    df_pheno['DateTime'] = [x.replace(',', '.') for x in df_pheno['DateTime']]
    
    
    df_pheno['DateTime'] = pd.to_numeric(df_pheno['DateTime'])
    
    #df_pheno['DateTime']=df_pheno['DateTime'].astype(float)
    time_list = df_pheno['DateTime']
    
    result = []
    for x in time_list:
        
        timestampconvert(x)
        result.append(timestampconvert(x))
    new_time_stamps = []
    for i in result:
        new_time_stamps.append((i-min(result)).total_seconds())
    df_pheno = df_pheno.fillna(0)
    df_pheno['DateTime'] = new_time_stamps
    # df_pheno
    return df_pheno

## Function for getting L1 and L2 lick events
def lick_event_calculate(csv_path):
    """
    This function will calcuate the lick sensor data from the phenosys csv files and return 3 lists of timestamps from sensor L1, L2 and the timestamps of both channels. It will also do a quick plotting for the lick sensor data for raster plots and the density plot of overall lick events.

    """
    df = datetime_convert_phenosys(csv_path)
    
    # Get L1 timestamps and MsgValue1
    L1_time = df[df['unitLabel']=='L1']['DateTime'].values
    L1_value = df[df['unitLabel']=='L1']['MsgValue1'].values

    L1_timestamps_new = []
    for idx,value in enumerate(L1_value):
        if len(value.split(','))>1:
            # print(value.split(','))
            for j in value.split(',')[1:]:
                # print(j.split('-')[0])
                individual_value = int(j.split('-')[0])*0.001 # convert ms to s
                # print(L1_time[idx]+individual_value)
                if idx == 0:
                    L1_timestamps_new.append(L1_time[idx]+individual_value)
                else:
                    L1_timestamps_new.append(L1_timestamps_new[-1]+individual_value)
        else:
            L1_timestamps_new.append(L1_time[idx])
    L1_timestamps_new = np.array(L1_timestamps_new)

    ## The same for L2
    L2_time = df[df['unitLabel']=='L2']['DateTime'].values
    L2_value = df[df['unitLabel']=='L2']['MsgValue1'].values

    L2_timestamps_new = []
    for idx,value in enumerate(L2_value):
        if len(value.split(','))>1:
            # print(value.split(','))
            for j in value.split(',')[1:]:
                # print(j.split('-')[0])
                individual_value = int(j.split('-')[0])*0.001
                # print(L1_time[idx]+individual_value)
                if idx == 0:
                    L2_timestamps_new.append(L2_time[idx]+individual_value)
                else:
                    L2_timestamps_new.append(L2_timestamps_new[-1]+individual_value)
        else:
            L2_timestamps_new.append(L2_time[idx])
    L2_timestamps_new = np.array(L2_timestamps_new)

    all_lick_events = np.array(sorted([*L1_timestamps_new,*L2_timestamps_new]))

    print('File processed: '+csv_path.split('/')[-1])
    print('Is there any duplication?: '+ str(len(L1_timestamps_new) != len(set(L1_timestamps_new))))

    fig, [ax0,ax1] = plt.subplots(nrows=2,ncols=1,sharex=True,figsize=[8,4])
    ax0.eventplot([L1_timestamps_new,L2_timestamps_new,all_lick_events],lw=0.5,linelengths=0.8,color=['C0','C1','C3'])
    legend=ax0.legend(['L1','L2','All'],bbox_to_anchor=(0., 1.1, 1., 1.1), loc=3, ncol=3, mode="expand", 
                borderaxespad=0.,frameon=False,title='Lick events for L1 & L2 lick sensors: '+csv_path.split('/')[-1],fontsize=8)
    
    legend.get_title().set_fontsize('9')
    
    # sns.distplot(L1_timestamps_new, hist=False, rug=True, ax=ax1, color='C0',rug_kws={"height":0.2,"linewidth":0.5,"alpha":0.5})
    # sns.distplot(L2_timestamps_new, hist=False, rug=True, ax=ax1, color='C1',rug_kws={"height":0.2,"linewidth":0.5,"alpha":0.5})
    sns.kdeplot(all_lick_events, ax=ax1, color='C3')
    sns.rugplot(all_lick_events, ax=ax1, color='C3',height=0.2, linewidth=0.5,alpha=0.5)
    ax1.set_title('All')
    ax0.set_yticks([])
    ax1.set_yticks([])
    plt.xlabel('Second',fontsize=8)
    plt.xticks(fontsize=8);plt.yticks(fontsize=8)
    plt.xlim([0,1600])
    sns.despine()
    plt.tight_layout();plt.show()
    return L1_timestamps_new, L2_timestamps_new, all_lick_events



## Function for getting P1 and P1C lick events
def GPIO_event_calculate(csv_path):
    """
    This function will calcuate the lick sensor data from the phenosys csv files and return 3 lists of timestamps from GPIOs P1 and P1C and the timestamps of both channels.

    """
    df = datetime_convert_phenosys(csv_path)
    
    # Get L1 timestamps and MsgValue1
    P1_time = df[df['unitLabel']=='P1']['DateTime'].values
    P1C_time = df[df['unitLabel']=='P1C']['DateTime'].values
    
    P2_time = df[df['unitLabel']=='P2']['DateTime'].values
    P2A_time = df[df['unitLabel']=='P2A']['DateTime'].values

    # all_pump_events = P1_time + P1C_time + P2_time + P2A_time
    pump_all = np.concatenate([P1_time,P1C_time,P2_time,P2A_time],axis=0)
    pump_all = sorted(pump_all)
    pump_all = np.array(pump_all)

    # fig, ax0 = plt.subplots(nrows=1,ncols=1,sharex=True,figsize=[8,2])
    # ax0.eventplot([P1_time,P1C_time,P2_time,P2A_time],lw=0.5,linelengths=0.8,color=['C0','C1','C3','C4'])
    # legend=ax0.legend(['P1','P1C','P2','P2A'],bbox_to_anchor=(0., 1.1, 1., 1.1), loc=3, ncol=4, mode="expand", 
    #             borderaxespad=0.,frameon=False,title='GPIO events: '+csv_path.split('/')[-1],fontsize=8)
    
    # legend.get_title().set_fontsize('9')
    
    # ax0.set_yticks([])
    # plt.xlabel('Second',fontsize=8)
    # plt.xticks(fontsize=8);plt.yticks(fontsize=8)
    # plt.xlim([0,1600])
    # plt.tight_layout();plt.show()
    
    return P1_time, P1C_time, P2_time, P2A_time, pump_all

## Filter function for np array
def filter_mask_large(arr, k):
    return arr[arr < k]
def filter_mask_small(arr, j):
    return arr[arr > j]

def filter_mask_range(arr, min, max):
    arr = filter_mask_large(arr, max)
    arr = filter_mask_small(arr, min)
    return arr

## find nearst value in np array
def find_nearest(array, value):
    array = np.asarray(array)
    idx = (np.abs(array - value)).argmin()
    return array[idx]

def select_pump(P1,P1C,P2,P2A):
  if (len(P1) + len(P1C)) > (len(P2) + len(P2A)):
    # print('Select P1/P1C')
    Pump1 = P1
    Pump2 = P1C
  else:
    # print('Select P2/2A')
    Pump1 = P2
    Pump2 = P2A
  return Pump1, Pump2

def remove_init(Pump1,Pump2):
  pump_all = [*Pump1,*Pump2]
  pump_all = sorted(pump_all)
  pump_all = np.array(pump_all)
  x_shift = pump_all[-1]
  x_bar = [x*180+300 for x in range(18)]
  x_bar = [0] + x_bar
  x_bar = np.array(x_bar)
  x_bar = x_bar[x_bar<x_shift]
  for x in x_bar:
    if len(Pump2) == 0: # again to prevent the 2nd pump is empty
      nearest_num = [find_nearest(Pump1,x)]
      if find_nearest(nearest_num,x) in Pump1:
        Pump1 = np.delete(Pump1, np.where(Pump1 == find_nearest(nearest_num,x)))
    else:
      nearest_num = [find_nearest(Pump1,x),find_nearest(Pump2,x)]
      if find_nearest(nearest_num,x) in Pump1:
        Pump1 = np.delete(Pump1, np.where(Pump1 == find_nearest(nearest_num,x)))
      elif find_nearest(nearest_num,x) in Pump2:    
        Pump2 = np.delete(Pump2, np.where(Pump2 == find_nearest(nearest_num,x)))
  Pump_all_no_init = [*Pump1,*Pump2]
  Pump_all_no_init = np.array(sorted(Pump_all_no_init))
  return Pump1,Pump2,Pump_all_no_init

def remove_GPIO_init(Pump1,Pump2):
  pump_all = [*Pump1,*Pump2]
  pump_all = sorted(pump_all)
  pump_all = np.array(pump_all)

  x_shift = pump_all[-1]
  x_bar = [x*180+300 for x in range(18)]
  # x_bar = [0] + x_bar
  x_bar = np.array(x_bar)
  x_bar = x_bar[x_bar<x_shift]
  for x in x_bar:
    if len(Pump2) == 0: # again to prevent the 2nd pump is empty
      nearest_num = [find_nearest(Pump1,x)]
      if find_nearest(nearest_num,x) in Pump1:
        Pump1 = np.delete(Pump1, np.where(Pump1 == find_nearest(nearest_num,x)))
    else:
      nearest_num = [find_nearest(Pump1,x),find_nearest(Pump2,x)]
      if find_nearest(nearest_num,x) in Pump1:
        Pump1 = np.delete(Pump1, np.where(Pump1 == find_nearest(nearest_num,x)))
      elif find_nearest(nearest_num,x) in Pump2:    
        Pump2 = np.delete(Pump2, np.where(Pump2 == find_nearest(nearest_num,x)))
  Pump_all_no_init = [*Pump1,*Pump2]
  Pump_all_no_init = np.array(sorted(Pump_all_no_init))
  return Pump1,Pump2,Pump_all_no_init


def altspace(start, step, count, endpoint=False, **kwargs):
   stop = start+(step*count)
   return np.linspace(start, stop, count, endpoint=endpoint, **kwargs)

In [ ]:
#@title Plotting setting 
## Plotting setting
plt.rcParams["font.family"] = "Arial"
plt.rcParams.update({'font.size': 10})
my_color_map = ['#56b4e9',
                '#e69f00',
                '#009e73',
                '#f0e442',
                '#0072b2',
                '#d55e00',
                '#cc79a7']

## For getting hex code from rgb
# rgb_code = [204,121,167]
# hex_color = '#%02x%02x%02x' % (rgb_code[0], rgb_code[1], rgb_code[2])
# print(hex_color)

In [ ]:
#@title Functions for plotting

def title_plot_name(mouse_id, Food_deprivation, date):

  if Food_deprivation == 'severe':
      title = mouse_id+' '+date+' '+Genotype+ ' Neurons (severe FD)'
  elif Food_deprivation == 'mild':
      title = mouse_id+' '+date+' '+Genotype+ ' Neurons (mild FD)'
  elif Food_deprivation == 'none':
      title = mouse_id+' '+date+' '+Genotype+ ' Neurons ($\it{ad}$ $\it{libitum}$)'
  plot_name = mouse_id+'_'+date+'_'+Genotype+'_'
  return title, plot_name

def length_delivery(Pump1,Pump2):
  deliver_milk = len(Pump1)
  deliver_water = len(Pump2)
  return deliver_milk, deliver_water

def GPIO_eventplot(Pump1, Pump2, ax):
  # f, ax = plt.subplots(figsize=[10,6])
  # y_offset = 0
  if len(Pump2) == 0:
    GPIO_plot = [Pump1]
    ax.eventplot(GPIO_plot,color=['C3'],lineoffsets=[y_offset*1.1],linewidths=0.5, linelength=1,linestyle = 'None',alpha=1)
  else:
    GPIO_plot = [Pump1,Pump2]
    ax.eventplot(GPIO_plot,color=[my_color_map[2],my_color_map[1]],lineoffsets=[1,0],linewidths=0.5, linelength=0.8,linestyle = 'None',alpha=1)
  return ax

def Ca2_eventplot(df_event_sorted,ax):
  for i,name in enumerate(df_event_sorted.columns):
    data=df_event_sorted[name][df_event_sorted[name] > 0]
    ax.eventplot(data.index.values,lineoffsets = i+1,linewidth=0.3,linelength=1,alpha=1,color='C0')#my_color_map[0])
  return ax

In [ ]:
## Save figs?
save_fig = False

In [ ]:
#functions for analysing 2P bulb data

def calciumpertake (file_date,mouse_id,iterator):
    #print (iterator)
    calciumpath='/Volumes/sdcard/forsuite2P/'+file_date+'/'+mouse_id+'/alltakes/suite2p/plane0/'
    isCell=np.load(calciumpath+'iscell.npy')
    Fluo=np.load(calciumpath+'F.npy')
    
    FNeu=np.load(calciumpath+'Fneu.npy')   #unclear why neuropil correction is done with F
    Ops=np.load(calciumpath+'ops.npy',allow_pickle=True).item()
    dfCorShift=pd.DataFrame(Ops['corrXY'])
    dfShiftx=pd.DataFrame(Ops['xoff']).transpose()
    dfShifty=pd.DataFrame(Ops['yoff']).transpose()
    dfisCell = pd.DataFrame(isCell)
    dfFluo=pd.DataFrame(Fluo)
    dfFNeu=pd.DataFrame(FNeu)
    dfFluoNeuCor=dfFluo-0.7*dfFNeu

    mean=dfCorShift.mean()
    #print(mean)
    max=0.5*np.max(dfCorShift)

    #print (max[0])
    dfCorShiftZ=(dfCorShift - dfCorShift.mean())/dfCorShift.std(ddof=0)
    dfFinalTrace=dfFluoNeuCor.loc[dfisCell[0]==1]#&(dfShift.iloc[[0]]==0)&(dfShift.iloc[[1]]==0)]
    
    #plt.plot(Ops['yoff'])
    #plt.plot(Ops['xoff'])
    print (dfFinalTrace.shape)

    filterx=dfShiftx != 0

    filtery=dfShifty != 0

    dfFinalTrace=np.where(dfShiftx != 0,np.nan,dfFinalTrace)

    dfFinalTrace=np.where(dfShifty != 0,np.nan,dfFinalTrace)

    dfFinalTrace=np.where(dfCorShift.transpose() < max[0],np.nan,dfFinalTrace)
    
    #print('huhu')
    SubTrace=dfFinalTrace[:,iterator*1200:iterator*1200+1200]
    dfSubTrace=pd.DataFrame(SubTrace)
    dfBaseline=dfSubTrace.T[dfSubTrace<dfSubTrace.T.quantile(q=0.25)].mean()
    dfSubTraceFF0=(dfSubTrace.T-dfBaseline)/dfBaseline
    #print (dfSubTraceFF0.shape)
    endProdukt=dfSubTraceFF0.T.to_numpy()
    #print (dfSubTrace)
    
    calciumypath='/Volumes/sdcard/forsuite2P/'+file_date+'/'+mouse_id+'/alltakes/yvec/'
    files_list = []
    for root, directories, files in os.walk(calciumypath):
        for name in files:
            files_list.append(os.path.join(root, name))
        #print(files_list)

    with open(files_list[0]) as json_file:
        data=json.load(json_file)
        yvalues=np.asarray(data['tvec'])
        yvalues=yvalues-yvalues[0]
        #print (yvalues)
    
    return endProdukt,yvalues



def pheno_event_perexp(csv_path):
    
    df = datetime_convert_phenosys(csv_path)
    
    result={} 
    
    
    listefastandslow=df.index[(df['MsgValue1']=='fastfor2P.xlsx')|(df['MsgValue1']=='slowfor2P.xlsx')].tolist()
    
    listetwoPLSM=df.index[df['unitLabel']=='2PLSM'].tolist()
    #print (len(listetwoPLSM))
    
    regex=re.compile("^P")
    regexb=re.compile("^L")
    listetwoPLSMTwo=listetwoPLSM.copy()
    for i in listetwoPLSM:
        j=int(i)+1
        
        if "exp" in df['unitLabel'].iloc[j] or regex.match(df['unitLabel'].iloc[j+1]) or "exp" in df['unitLabel'].iloc[j+1]or "exp" in df['unitLabel'].iloc[j+2]:
            ##search for P plus number with regex!!
            print("save"+str(i))
            
        else:    
            listetwoPLSMTwo.remove(i)
            print ('delete'+str(i))
        
        
        
        
    listeendExps=df.index[df['SystemMsg']=='exp end'].tolist()
    listetwoPLSMTwo.append(str(len(df)))
    #print (len(listetwoPLSMTwo))
    #print (len(listeendExps))
    #print (len(listefastandslow))
    #print (listetwoPLSMTwo)
    #print (listetwoPLSM)
    dfExpsSorted=pd.read_excel("/Users/friedrichjohenning/Dropbox/phenosyspermouse/overviewkeptexps.xlsx", sheet_name=mouse_id,na_values='--')
    
    indexer=0
    #print (listetwoPLSMTwo)
    
    for i in listetwoPLSMTwo:
        i=int(i)
        #print(i)
        #print ()
        
        if int(i)< len(df):
            j=int(listetwoPLSMTwo[indexer+1])
        else:
            break
         
        
        #print (dfExpsSorted[int(file_date)][indexer])
        
        if dfExpsSorted[int(file_date)][indexer]==0:
            print ("fef")
            #print (indexer)
            indexer=indexer+1
            
            continue
        
        #P1_position = df.loc[int(i):,'unitLabel']=='P1'.filter(items=[int(i),int(i)+5], axis=0)
        
        
        df2=df.iloc[int(i):int(j)]
        
        
        P1_time = df2[df2['unitLabel']=='P1']['DateTime'].values
        P1C_time= df2[df2['unitLabel']=='P1C']['DateTime'].values
        L1_time= df2[df2['unitLabel']=='L1']['DateTime'].values
        
        
        #if time course should be preserved
        #if i== listetwoPLSMTwo[0]:
           #diff=P1_time[0]
        
        #zero for every exp run
        #diff=P1_time[0]
        diff=df2[df2['unitLabel']=='2PLSM']['DateTime'].values[0]
        
        P1_time = P1_time - diff
        P1_time=P1_time[(P1_time<90)&(P1_time>0)]
        P1C_time = P1C_time - diff
        P1C_time=P1C_time[(P1C_time<90)&(P1C_time>0)]
        L1_time = L1_time - diff
        L1_time=L1_time[(L1_time<90)&(L1_time>0)]
        
        #print (P1_time.shape)
        if P1_time.shape[0]>1:
            
            latency=P1_time[1]-P1_time[0]
            
        else: 
            latency=None
            
        #print("kuckuck") 
        #print(indexer)
        #print (dfExpsSorted[int(file_date)][indexer])
        
            
        if (dfExpsSorted[int(file_date)][indexer])== 1:
            
            suffix="_slow"
            
        elif (dfExpsSorted[int(file_date)][indexer])== 2:
            suffix="_fast"
    
        else:
            suffix="_aka"    
            
        indexer=indexer+1
    
    
        
        result[str(i)+str(df['MsgValue1'].iloc[int(i)-1])+suffix]=[P1_time.copy(),P1C_time.copy(),L1_time.copy(),latency]
    
    
        
    
    
    
    return result
    
    

# Generate dictionaries from raw data and upload raw data

In [ ]:
## get folder
# modify this line for your local path
# file_path = 'local path'
Phenosys_folder = '/Users/friedrichjohenning/Dropbox/phenosyspermouse/allexps/'
extension = 'csv'
os.chdir(Phenosys_folder)
result = sorted(glob.glob('*.{}'.format(extension)))
print(result)

In [ ]:
# generates dictionary (mouse: mouse_id:file_date:individualexperiment)for the single exps with calcium traces and behavior according to individual takes
dates=[]
mouse={}
days={}
singleExps={}

mouselist=["SNA095265","SNA095266","SNA095267","SNA095270"]


for mouse_id in mouselist:

    selected_result = [r for r in result if mouse_id in r]
    print (selected_result)

    for idx,r in enumerate(tqdm(selected_result[:])):
        #print (r)  
        file_date = r.split('-')[1].split('.')[0] + r.split('-')[1].split('.')[1] + r.split('-')[1].split('.')[2]
        print (file_date)  
        #file_date = '211116'
        dates.append(file_date)

    days=dict.fromkeys(dates)
    mouse[mouse_id]=days

    for idx,r in enumerate(tqdm(selected_result[:])):
        print (r)  
        file_date = r.split('-')[1].split('.')[0] + r.split('-')[1].split('.')[1] + r.split('-')[1].split('.')[2]
        #file_date ='211116'
        mouse[mouse_id][file_date]=pheno_event_perexp(Phenosys_folder + r)
        #print (len(mouse[mouse_id][file_date].keys()))
        print (mouse[mouse_id][file_date].keys())
        i=0
        for key, value in mouse[mouse_id][file_date].items():
            #print ('file_date')
            #print (file_date)
            #print(len(mouse[mouse_id][file_date]))
            #print('key')
            #print(key)
            #print('i')
            #print (i)
            #print(value)

            calciumtraces=calciumpertake(file_date,mouse_id,i)

            value.append([calciumtraces[0],calciumtraces[1],"hoho"])
            #plt.plot(calciumtraces)
            i+=1


        if mouse[mouse_id][file_date]==None:
            mouse[mouse_id].pop([file_date])


In [ ]:
#generates dictionary (dictofdataframes: mouse id: exp date: dataframe) of concatenated daily behavior exps and cells 

dictofdataframes={}

ypsilon=np.vstack(np.arange(0,119.9,0.1))



for key,value in mouse.items():
    mouseId=key
    print (mouseId)
    dataFrames={}
    
    for keya,valuea in value.items():
        expDate=keya
        print (expDate)
          
        i=0 
        pumps=pd.Series()
        emptypumps=pd.Series()
        licks=pd.Series()
        calciumx=pd.Series()
        calciumy=pd.DataFrame()
        
        for keyc,valuec in valuea.items():
            
            
            expId=keyc
            print (expId)
            
            if valuec[4][0].shape[0]==0 or valuec[4][0].shape[1]==0:
                print('continue')
                continue
                
            print (valuec[0].shape)
            pumps=pumps.append(pd.Series(valuec[0]+i*120), ignore_index=True)
            
            emptypumps=emptypumps.append(pd.Series(valuec[1]+i*120), ignore_index=True)
            
            
            
            print (valuec[2].shape)
            licks=licks.append(pd.Series(valuec[2]+i*120), ignore_index=True)
            
            calciumx=calciumx.append(pd.Series(valuec[4][1]+i*120), ignore_index=True)
            print (valuec[4][0].shape[0])
            print (type(valuec[4][0]))
            
            calciumy=calciumy.append(pd.DataFrame(valuec[4][0].T), ignore_index=True)
            i=i+1
            
        dataFrames[expDate]=[pumps,licks,calciumx,calciumy,emptypumps]       
                
        dictofdataframes[mouseId]= dataFrames   

In [ ]:
#generates PSTHs of milk and milkbinge and empty licks based on dictofdataframes

PSTH_trace_milk = {} 
PSTH_trace_milkbinge = {}
PSTH_trace_empty = {} 


num_bins = 100
t_scale = np.linspace(-1,4,100)


for key,value in dictofdataframes.items():
    
    mouse_id=key
    print (mouse_id)
    dfExpsSorted=pd.read_excel("/Users/friedrichjohenning/Dropbox/phenosyspermouse/overviewkeptexps.xlsx", sheet_name=mouse_id,na_values='--')
    
    for key, value in value.items():
        exp_date=key
        print(exp_date)
        slow_deliveries_milk=pd.Series()
        slow_deliveries_empty=pd.Series()
        fast_deliveries_milk=pd.Series()
        df_z=value[3].copy()
        df_z=df_z.set_index(value[2])
        liste=[]
        
        listebinge=[]
        
        
        try: 
            
            for i in dfExpsSorted[int(exp_date)].index:
                
                if dfExpsSorted[int(exp_date)][i] == 1:
                    liste.append(i)
                if dfExpsSorted[int(exp_date)][i] == 2:
                    listebinge.append(i)    
            
            print (liste)
            
            for i in liste:
                print (i)
                
                dftrans=value[0][(value[0] > i*120) & (value[0] < (i+1)*120)] 
                dftranse=value[4][(value[4] > i*120) & (value[4] < (i+1)*120)]  
                slow_deliveries_empty=slow_deliveries_empty.append(dftranse,ignore_index=True )
                slow_deliveries_milk=slow_deliveries_milk.append(dftrans,ignore_index=True )
                
            
            
            for i in listebinge:
                print (i)
                
                dftrans=value[0][(value[0] > i*120-6) & (value[0] < (i+1)*120)]
                boutinit=dftrans[(dftrans-dftrans.shift()>4)&(dftrans-dftrans.shift(-1)<4)&(dftrans-dftrans.shift(-2)<4+dftrans-dftrans.shift(-1))]
                print(boutinit)
                fast_deliveries_milk=fast_deliveries_milk.append(boutinit,ignore_index=True )
            
            
            data = df_z.copy()
            cellset = data.columns
            ## generate neural matrix in dict
            for cell in cellset:
                #print(cell)
                print (len(data[cell])) 
                PSTH_trace = pd.DataFrame()
                for trial, time in enumerate(slow_deliveries_milk):
                    #print (trial)
                    times=time-1
                    timee=time+4
                    #print (times)
                    #print (timee)

                    PSTH=data[cell][(data[cell].index>times) & (data[cell].index<timee)]
                    PSTH.index=np.arange(0,PSTH.shape[0]/10,0.10)
                    #print (PSTH.shape[0])
                    #print (PSTH.index)
                    frames=[PSTH_trace,PSTH]
                    PSTH_trace=pd.concat(frames,axis=1,ignore_index=True)
                PSTH_trace_milk[str(mouse_id)+str(exp_date)+str(cell)] = PSTH_trace
                
                PSTH_traceempty = pd.DataFrame()
                for trial, time in enumerate(slow_deliveries_empty):
                    #print (trial)
                    times=time-1
                    timee=time+4
                    #print (times)
                    #print (timee)

                    PSTHe=data[cell][(data[cell].index>times) & (data[cell].index<timee)]
                    PSTHe.index=np.arange(0,PSTHe.shape[0]/10,0.10)
                    #print (PSTH.shape[0])
                    #print (PSTH.index)
                    frames=[PSTH_traceempty,PSTHe]
                    PSTH_traceempty=pd.concat(frames,axis=1,ignore_index=True)
                PSTH_trace_empty[str(mouse_id)+str(exp_date)+str(cell)] = PSTH_traceempty

                PSTH_tracefast = pd.DataFrame()
                for trial, time in enumerate(fast_deliveries_milk):
                    #print (trial)
                    times=time-1
                    timee=time+4
                    #print (times)
                    #print (timee)

                    PSTHf=data[cell][(data[cell].index>times) & (data[cell].index<timee)]
                    PSTHf.index=np.arange(0,PSTHf.shape[0]/10,0.10)
                    #print (PSTH.shape[0])
                    #print (PSTH.index)
                    frames=[PSTH_tracefast,PSTHf]
                    PSTH_tracefast=pd.concat(frames,axis=1,ignore_index=True)
                PSTH_trace_milkbinge[str(mouse_id)+str(exp_date)+str(cell)] = PSTH_tracefast



            
        except KeyError:
            
            #print ("huhu")
            continue
            
            
            
        
    

# AUROC


In [ ]:
#auroc slow milk
from sklearn.metrics import auc

'''# Downsample the temporal resolution to speed up computing
PSTH_milk_binned = np.zeros((PSTH_milk.shape[0],PSTH_milk.shape[1],50)) # generate a empty np array to host downsampled neural array
for idx in range(PSTH_milk.shape[0]):
  for trial in range(PSTH_milk.shape[1]):
    PSTH_milk_binned[idx,trial,:] = down_sample(PSTH_milk[idx,trial,:],f=2) # down sample from 20 Hz to 10 hz, you can skip this part
'''

#0 index of neuron: dictionary 
#1  trial
#2 time
auc_total_all_cell = {}

for key,value in PSTH_trace_milk.items():
    
    #print (key)
    #print (value.shape[0])
    baseline = value[0:9].to_numpy().flatten() # take first 10 time bin (-1 to 0 sec) from each trials as baseline 
    auc_total = []
    for i in range(value.shape[0]): # walk through each time bin
        
        stimulus = value.iloc[i,:]
        pool = [*baseline, *stimulus]
        steps = 50
        criteria = [min(pool) + (j*(max(pool)-min(pool))/steps) for j in range(steps)]
        criteria[0] = criteria[0] - 1e-12 # modify the min value to slighter lesser value, so baseline/stimulus will be larger than the min of criteria
        pbase = []
        pstim = []
        # Calculate P(above threshold) for each cutoff
        for cri in criteria:
            pbase.append(sum(baseline > cri)/len(baseline))
            pstim.append(sum(stimulus > cri)/len(stimulus))
        auc_ = auc(x = pbase, y = pstim)
        auc_total.append(auc_)
    auc_total_all_cell[key] = auc_total   
        

In [ ]:
#auroc binging
from sklearn.metrics import auc

'''# Downsample the temporal resolution to speed up computing
PSTH_milk_binned = np.zeros((PSTH_milk.shape[0],PSTH_milk.shape[1],50)) # generate a empty np array to host downsampled neural array
for idx in range(PSTH_milk.shape[0]):
  for trial in range(PSTH_milk.shape[1]):
    PSTH_milk_binned[idx,trial,:] = down_sample(PSTH_milk[idx,trial,:],f=2) # down sample from 20 Hz to 10 hz, you can skip this part
'''

#0 index of neuron: dictionary 
#1  trial
#2 time
auc_total_all_cellbinge = {}

for key,value in PSTH_trace_milkbinge.items():
    
    #print (key)
    #print (value.shape[0])
    baseline = value[0:9].to_numpy().flatten() # take first 10 time bin (-1 to 0 sec) from each trials as baseline 
    auc_total = []
    for i in range(value.shape[0]): # walk through each time bin
        
        stimulus = value.iloc[i,:]
        pool = [*baseline, *stimulus]
        steps = 50
        criteria = [min(pool) + (j*(max(pool)-min(pool))/steps) for j in range(steps)]
        criteria[0] = criteria[0] - 1e-12 # modify the min value to slighter lesser value, so baseline/stimulus will be larger than the min of criteria
        pbase = []
        pstim = []
        # Calculate P(above threshold) for each cutoff
        for cri in criteria:
            pbase.append(sum(baseline > cri)/len(baseline))
            pstim.append(sum(stimulus > cri)/len(stimulus))
        auc_ = auc(x = pbase, y = pstim)
        auc_total.append(auc_)
    auc_total_all_cellbinge[key] = auc_total   
        

# Save dictionaries

In [ ]:
savedicti(auc_total_all_cell,"aurocslow")
savedicti(auc_total_all_cellbinge,"aurocfast")

In [ ]:
#savedicti(auc_total_all_cell,"aurocslow")
#savedicti(auc_total_all_cellbinge,"aurocfast")
savedicti(PSTH_trace_milk,"PSTHslow")
savedicti(PSTH_trace_milkbinge,"PSTHfast")
savedicti(PSTH_trace_empty,"PSTHempty")

savedicti(mouse,"everysingleexp")
savedicti(dictofdataframes, "concexpdayspermouse")


# figures (code dump)

In [ ]:
#generates and plots behavior and all calcium trces of individual sweeps, not concatenated.
pump_num=[]        
for key,value in mouse.items():
    mouseId=key
    
    for key,value in value.items():
        expDate=key
        #print (key)
        for key,value in value.items():
            expId=key 
            if value[0].shape[0]>=1:
                pump_num.append(value[0].shape[0]-1)
            else:
                pump_num.append(value[0].shape[0])




ypsilon=np.vstack(np.arange(0,119.9,0.1))


for key,value in mouse.items():
    mouseId=key
    for keya,valuea in value.items():
        expDate=keya
        for keyb,valueb in valuea.items():
            expId=keyb 
            if valueb[0].shape[0]>=1:
                pump_num.append(valueb[0].shape[0]-1)
            else:
                pump_num.append(valueb[0].shape[0])
        for keyc,valuec in valuea.items():
            expId=keyc
            #print (valuec[0].shape)
            if valuec[4][0].shape[0]==0 or valuec[4][0].shape[1]==0:
                print('continue')
                continue
                
            print (idx)    
            #print (valuec[4][0])
            fig, ax = plt.subplots(figsize=[10,4],ncols=2,nrows=3,gridspec_kw={'width_ratios':[4,1]})
            ax[0,0].set_title(mouseId+" "+expDate+" "+expId)
            ax[0,0].eventplot(valuec[0],linelengths = 0.8,linewidths=0.6,lineoffsets = idx,color=my_color_map[0])
            ax[0,0].text(s=file_date,x=275,y=idx,va='center',ha='right')
            sns.despine(left=True)
            ax[0,0].set_yticks([])
            ax[0,0].set_xlim([0,120])
            ax[1,0].eventplot(valuec[2],linelengths = 0.8,linewidths=0.6,lineoffsets = idx,color=my_color_map[1])
            ax[1,0].text(s=file_date,x=275,y=idx,va='center',ha='right')
            sns.despine(left=True)
            ax[1,0].set_yticks([])
            ax[1,0].set_xlim([0,120])
            
            ax[2,0].plot(valuec[4][1],valuec[4][0].T,linewidth=0.2)
            ax[2,0].set_xlim([0,120])
            ax[2,0].set_ylim([-0.05,0.1])
            
            """ax[2,0]=sns.heatmap(valuec[4][0].T)"""
            
                
      

In [ ]:
#plots concatenated behavior and all cells underneath for every day for one mouse
mouse_Id="SNA095265"
for key, values in dictofdataframes[mouse_id].items():
    print (mouse_Id)
    print (key)
    print (dataFrames[key][3].shape[1])
    zellen=dataFrames[key][3].shape[1]
    fig, ax = plt.subplots(figsize=[14,28],ncols=1,nrows=2+zellen,sharex=True)
    ax[0].set_title(key)
    ax[0].eventplot(dataFrames[key][0],linelengths = 0.2,linewidths=0.1,lineoffsets = idx,color=my_color_map[0])
    ax[0].eventplot(dataFrames[key][4],linelengths = 0.2,linewidths=0.1,lineoffsets = idx,color=my_color_map[5])
    #ax[0].text(s=file_date,x=275,y=idx,va='center',ha='right')
    sns.despine(left=True)
    ax[0].set_yticks([])
    #ax[0,0].set_xlim([0,120])
    ax[1].eventplot(dataFrames[key][1],linelengths = 0.2,linewidths=0.1,lineoffsets = idx,color=my_color_map[1])
    #ax[1].text(s=file_date,x=275,y=idx,va='center',ha='right')
    sns.despine(left=True)
    ax[1].set_yticks([])
    #ax[1,0].set_xlim([0,120])
    for n in range(zellen):
        ax[2+n].plot(dataFrames[key][2],dataFrames[key][3][n],linewidth=0.2)
        
        #ax[2+n].set_ylim([-0.05,0.1])


In [ ]:
#plots behavior above each cell
for key, values in dataFrames.items():
    print (key)
    print (dataFrames[key][3].shape[1])
    zellen=dataFrames[key][3].shape[1]
    
    for n in range(zellen):
            fig, ax = plt.subplots(figsize=[14,4],ncols=1,nrows=3,sharex=True)
            ax[0].set_title(key+str(n))
            ax[0].eventplot(dataFrames[key][0],linelengths = 0.2,linewidths=0.5,lineoffsets = idx,color=my_color_map[0])
            #ax[0].text(s=file_date,x=275,y=idx,va='center',ha='right')
            sns.despine(left=True)
            ax[0].set_yticks([])
            #ax[0,0].set_xlim([0,120])
            ax[1].eventplot(dataFrames[key][1],linelengths = 0.2,linewidths=0.1,lineoffsets = idx,color=my_color_map[1])
            #ax[1].text(s=file_date,x=275,y=idx,va='center',ha='right')
            sns.despine(left=True)
            ax[1].set_yticks([])
            ax[2].plot(dataFrames[key][2],dataFrames[key][3][n],linewidth=0.2)
        
        #ax[2+n].set_ylim([-0.05,0.1])

In [ ]:
#plot of example cell from labseminar
fig, ax = plt.subplots(figsize=[14,4],ncols=1,nrows=3,sharex=True)
#ax[0].set_title(key+str(n))
ax[0].eventplot(dataFrames["211123"][0],linelengths = 0.2,linewidths=0.5,lineoffsets = idx,color=my_color_map[0])
#ax[0].text(s=file_date,x=275,y=idx,va='center',ha='right')
sns.despine(left=True)
ax[0].set_yticks([])
#ax[0,0].set_xlim([0,120])
ax[1].eventplot(dataFrames["211123"][1],linelengths = 0.2,linewidths=0.1,lineoffsets = idx,color=my_color_map[1])
#ax[1].text(s=file_date,x=275,y=idx,va='center',ha='right')
sns.despine(left=True)
ax[1].set_yticks([])
ax[2].plot(dataFrames["211123"][2],dataFrames["211123"][3][8],linewidth=0.2)

fig.savefig('/Users/friedrichjohenning/Desktop/figurehung/211123570roi9.pdf')
        

In [ ]:
fig, axes = plt.subplots(nrows=1,ncols=2,figsize=[7,16])

new_df = pd.DataFrame()
new_dfbinge = pd.DataFrame()
for key in PSTH_trace_milk.keys():
    print(key)
    new_df[key]=(PSTH_trace_milk[key].mean(axis=1)) 
    # get average traces from each neurons/keys
#for key in PSTH_trace_milkbinge.keys():
    #print(key)
    new_dfbinge[key]=(PSTH_trace_milkbinge[key].mean(axis=1))
bsorted=(new_dfbinge[9:48].mean(axis=0)-new_dfbinge[0:8].mean(axis=0))*-1
#bsorted=(new_df[9:48].mean(axis=0)-new_df[0:8].mean(axis=0))*-1


sorted=bsorted.sort_values()

sorter=list(sorted.index)
   
isort = new_df.reindex(columns=sorter) # sort neuron index with responses 0 to +2 sec upon milk delivery
isortbinge=new_dfbinge.reindex(columns=sorter)
axes[0].imshow(isort.T,cmap='viridis',aspect='auto',vmin=-0.001,vmax=0.03)
axes[0].axvline(x=10,ls=':',color='k')
axes[0].set_title('time from pump activation')
axes[0].set_ylabel('neurons')
axes[0].set_xlabel('time bins (100 ms)')
axes[1].imshow(isortbinge.T,cmap='viridis',aspect='auto',vmin=-0.001,vmax=0.03)
axes[1].axvline(x=10,ls=':',color='k')
axes[1].set_title('time from pump activation')
axes[1].set_ylabel('neurons')
axes[1].set_xlabel('time bins (100 ms)')



In [ ]:
y=(new_dfbinge[9:48].mean(axis=0)-new_dfbinge[0:8].mean(axis=0))*-1 
x=bsorted=(new_df[9:48].mean(axis=0)-new_df[0:8].mean(axis=0))*-1  # 

In [ ]:
fig=plt.gcf()
fig.set_size_inches(4, 4)
sns.scatterplot(x=x,y=y,alpha=0.3)
sns.lineplot(x=(0,0),y=(-0.05,0.05),color="grey",linewidth=0.5)
sns.lineplot(x=(-0.05,0.05),y=(0,0),color="grey",linewidth=0.5)

plt.xlim(-0.05,0.05)
plt.ylim(-0.05,0.05)



In [ ]:
savedicti(auc_total_all_cell,"aurocslow")
savedicti(auc_total_all_cellbinge,"aurocfast")
savedicti(PSTH_trace_milk,"PSTHslow")
savedicti(PSTH_trace_milkbinge,"PSTHfast")

In [ ]:
fig, axes = plt.subplots(nrows=1,ncols=2,figsize=[7,4])

new_df = pd.DataFrame()
new_dffast = pd.DataFrame()

for key in aurocslow.keys():
    #print (key)
    #print (type(aurocslow[key]))
    if len(aurocslow[key])<50:
        
        aurocslow[key].extend(np.zeros(50-len(aurocslow[key])))
        
    if len(aurocslow[key])==0:
       
        aurocslow[key]=np.zeros(50)
           
    
    
    
    new_df[key]=np.asarray(aurocslow[key]) 
    
for key in aurocfast.keys():
    #print (key)
    #print (type(aurocslow[key]))
    if len(aurocfast[key])<50:
        
        aurocfast[key].extend(np.zeros(50-len(aurocfast[key])))
        
    if len(aurocfast[key])==0:
       
        aurocfast[key]=np.zeros(50)
           
    
    
    
    new_dffast[key]=np.asarray(aurocfast[key]) 
        
    
    
isortnew=new_df.reindex(columns=idx_posnodup)    
bsorted=(isortnew[10:48].mean(axis=0))*-1  # -new_df[0:8].mean(axis=0)


sorted=bsorted.sort_values()

sorter=list(sorted.index)
   
isort = isortnew.reindex(columns=sorter)
liste=list(isort.columns)
isortfast=new_dffast.reindex(columns=liste)
isort.index=np.arange(0,5,0.1)
isortfast.index=np.arange(0,5,0.1)


axes[0].imshow(isort.T,cmap='viridis',aspect='auto',vmin=0,vmax=1)
axes[0].axvline(x=10,ls=':',color='k')
axes[0].set_title('time from pump activation')
axes[0].set_ylabel('neurons')
axes[0].set_xlabel('time bins (100 ms)')

axes[1].imshow(isortfast.T,cmap='viridis',aspect='auto',vmin=0,vmax=1)
axes[1].axvline(x=10,ls=':',color='k')
axes[1].set_title('time from pump activation')
axes[1].set_ylabel('neurons')
axes[1].set_xlabel('time bins (100 ms)')

fig.savefig('/Users/friedrichjohenning/Desktop/figurehung/overviewPTSH.pdf')
    
    # get average traces from each neurons/keys
#for key in PSTH_trace_milkbing'''

In [ ]:

idx_positive = 


for key,value in aurocslow.items():
    
    

    data_auc = value.copy()
      
    data_aunp=np.asarray(data_auc)
    print (type(data_aunp))
    
    if len(data_aunp)<31:
        data_aunp=np.zeros(50)
    
    print (len(data_aunp))
    threshold_high = data_aunp[0:9].mean() + data_aunp[0:9].std()*3.5
      
    j_temp = []
    data_temp = []
    print(key)
    
    for j in range(10,30):  # Only look at first 2 sec after delivery
        if data_aunp[j] > threshold_high:
            j_temp.append(j)
            print (len(j_temp))
            
        if len(j_temp) > 3:
            
            print(np.diff(j_temp))
            
            if checkConsecutive(j_temp,n=4) == True:
                
                idx_positive.append(key)
          # print('# {} is responding to milk'.format(idx))

In [ ]:
callable(j_temp)

# coderia

In [ ]:
for key, value in auc_total_all_cell.items():
    print (key)
    plt.plot(value)
    plt.ylim(0,1)
    plt.show()

In [ ]:
fig, axes = plt.subplots(nrows=1,ncols=2,figsize=[7,16])

new_df = pd.DataFrame()
new_dfempty = pd.DataFrame()
for key in PSTH_trace_milk.keys():
    print(key)
    new_df[key]=(PSTH_trace_milk[key].mean(axis=1)) 
    # get average traces from each neurons/keys
#for key in PSTH_trace_milkbinge.keys():
    #print(key)
for key in PSTH_trace_empty.keys():
    print(key)    
    new_dfempty[key]=(PSTH_trace_empty[key].mean(axis=1))
bsorted=(new_df[9:19].mean(axis=0)-new_dfbinge[0:8].mean(axis=0))*-1
sorted=bsorted.sort_values()

sorter=list(sorted.index)
   
isort = new_df.reindex(columns=sorter) # sort neuron index with responses 0 to +2 sec upon milk delivery
isortempty=new_dfempty.reindex(columns=sorter)
axes[0].imshow(isort.T,cmap='viridis',aspect='auto',vmin=-0.001,vmax=0.03)
axes[0].axvline(x=10,ls=':',color='k')
axes[0].set_title('time from pump activation')
axes[0].set_ylabel('neurons')
axes[0].set_xlabel('time bins (100 ms)')
axes[1].imshow(isortempty.T,cmap='viridis',aspect='auto',vmin=-0.001,vmax=0.03)
axes[1].axvline(x=7.5,ls=':',color='k')
axes[1].set_title('time from pump activation')
axes[1].set_ylabel('neurons')
axes[1].set_xlabel('time bins (100 ms)')

In [ ]:
len(sorted)

In [ ]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

In [ ]:
new_dfempty

In [ ]:
onlys=new_df[new_dfbinge.isna().any()]
onlys

In [ ]:
df_z=dataFrames["211123"][3].copy()
df_z=df_z.set_index(dataFrames["211123"][2])


In [ ]:
dfExpsSorted=pd.read_excel("/Users/friedrichjohenning/Dropbox/phenosyspermouse/overviewkeptexps.xlsx", sheet_name=mouse_id,na_values='--')

dfExpsSorted[211123]
liste=[]
for i in dfExpsSorted[211123].index:
    if dfExpsSorted[211123][i] == 1:
        liste.append(i)

    

In [ ]:
liste

In [ ]:

slow_deliveries_milk=pd.Series()
for i in liste:
    dftrans=dataFrames["211123"][0][(dataFrames["211123"][0] > i*120) & (dataFrames["211123"][0] < (i+1)*120)]
    
    slow_deliveries_milk=slow_deliveries_milk.append(dftrans,ignore_index=True )
    
    

In [ ]:
PSTH_trace_milk = {} 
PSTH_trace_water = {} 
data = df_z.copy()


cellset = data.columns

num_bins = 100
t_scale = np.linspace(-1,4,100)

# np.random.seed(0)

## generate neural matrix in dict
for cell in cellset:
    #print(cell)
    print (len(data[cell])) 
    PSTH_trace = pd.DataFrame()
    for trial, time in enumerate(slow_deliveries_milk):
        print (trial)
        times=time-1
        timee=time+4
        print (times)
        print (timee)
        
        PSTH=data[cell][(data[cell].index>times) & (data[cell].index<timee)]
        PSTH.index=np.arange(0,PSTH.shape[0]/10,0.101)
        print (PSTH.shape[0])
        print (PSTH.index)
        frames=[PSTH_trace,PSTH]
        PSTH_trace=pd.concat(frames,axis=1,ignore_index=True)
    PSTH_trace_milk[cell] = PSTH_trace
    


In [ ]:
df_z

In [ ]:
plt.plot(PSTH_trace_milk[9].mean(axis=1))
new_df = np.array([PSTH_trace_milk.get(key) for key in PSTH_trace_milk.keys()]).mean(axis=1)

In [ ]:
isort

In [ ]:
new_df

In [ ]:
bsorted=new_df[7:17].mean(axis=0)
sorted=bsorted.sort_values()
sorted.index

In [ ]:
new_df 

In [ ]:
  dataFrames["211123"][1][dataFrames["211123"][1] > 4*120]

In [ ]:
mouse 

In [ ]:
pd.options.display.max_rows =6000
pd.options.display.max_columns = 120

In [ ]:
mouse_id='SNA095266'
file_date='211116'

In [ ]:
calciumypath='/Volumes/sdcard/forsuite2P/'+file_date+'/'+mouse_id+'/alltakes/yvec/'
files_list = []
for root, directories, files in os.walk(calciumypath):
   for name in files:
      files_list.append(os.path.join(root, name))
print(files_list)

with open(files_list[0]) as json_file:
    data=json.load(json_file)
    yvalues=np.asarray(data['tvec'])
    yvalues=yvalues-yvalues[0]
    print (yvalues)

In [ ]:
calciumpath='/Volumes/sdcard/forsuite2P/'+file_date+'/'+mouse_id+'/alltakes copy/suite2p/plane0/'
isCell=np.load(calciumpath+'iscell.npy')
Fluo=np.load(calciumpath+'F.npy')
FNeu=np.load(calciumpath+'F.npy')
Ops=np.load(calciumpath+'ops.npy',allow_pickle=True).item()
dfCorShift=pd.DataFrame(Ops['corrXY'])
dfShiftx=pd.DataFrame(Ops['xoff']).transpose()
dfShifty=pd.DataFrame(Ops['yoff']).transpose()


dfbadframes=pd.DataFrame(Ops['badframes'])
dfisCell = pd.DataFrame(isCell)
dfFluo=pd.DataFrame(Fluo)
dfFNeu=pd.DataFrame(FNeu)
dfFluoNeuCor=dfFluo-0.7*dfFNeu

mean=dfCorShift.mean()
print(mean)
max=0.5*np.max(dfCorShift)

print (max[0])
dfCorShiftZ=(dfCorShift - dfCorShift.mean())/dfCorShift.std(ddof=0)
dfFinalTrace=dfFluoNeuCor.loc[dfisCell[0]==1]#&(dfShift.iloc[[0]]==0)&(dfShift.iloc[[1]]==0)]
#plt.plot(Ops['yoff'])
#plt.plot(Ops['xoff'])
#dfFinalTrace

filterx=dfShiftx != 0

filtery=dfShifty != 0

dfFinalTrace=np.where(dfShiftx != 0,np.nan,dfFinalTrace)

dfFinalTrace=np.where(dfShifty != 0,np.nan,dfFinalTrace)

dfFinalTrace=np.where(dfCorShift.transpose() < max[0],np.nan,dfFinalTrace)

In [ ]:
Ops

In [ ]:
dfCorShiftB=np.where(dfbadframes == False,np.nan,dfCorShift)
dfCorShiftC=np.where(dfCorShift < max[0],np.nan,dfCorShift)

In [ ]:
plt.plot(dfCorShift)
plt.plot(dfCorShiftB)
plt.plot(dfCorShiftC)


plt.xlim(1200,7200)
plt.show()
plt.plot(Ops['yoff'])
plt.plot(Ops['xoff'])

plt.xlim(200,750)
plt.show()
plt.plot(dfFinalTrace.transpose())
plt.plot(dfbadframes)
#plt.xlim(2600,2900)
#plt.ylim(4500,7500)
#plt.xlim(200,750)


In [ ]:
plt.plot(dfCorShift)
plt.plot(dfCorShiftB)
plt.plot(dfCorShiftC)


#plt.xlim(200,2000)
plt.show()
plt.plot(Ops['yoff'])
plt.plot(Ops['xoff'])

plt.xlim(200,750)
plt.show()
plt.plot(dfFinalTrace.transpose())
plt.plot(dfbadframes)
#plt.xlim(2600,2900)
#plt.ylim(4500,7500)
#plt.xlim(200,750)



In [ ]:
dfbadframes

In [ ]:
## Manually select which date to view
# date = '201002'
date = '201002'
lick_sensor_align = True    # If True, the script will find the first activation pump at 5 mins and align it with the Phenosys csv where lick sensor data is available preserved. 
                            # If False, the script will transform GPIO data into timestamps and use it directly for general plotting and timestamps for milk/water deliveries.

## Select Ca2+ traces by mouse id and date
selected_result_ca = [r for r in result_ca if mouse_id in r]
selected_result_ca = [r for r in selected_result_ca if date in r]
print(selected_result_ca)

## Set date for phenosys csv
date_pheno = date[0:2]+'.'+date[2:4]+'.'+date[4:6]

In [ ]:
## Now align the Ca2+ data with the time difference.

#@title Load selected csv file

## Load Ca2+ dF/F data
from scipy import stats
r_path_ca = [r for r in selected_result_ca if 'celltraces.csv' in r]
print(r_path_ca)
dff_path = Ca2_folder + r_path_ca[0]
df_all = pd.read_csv(dff_path,header=[0,1],index_col=0)
# Selecet only the accepted cells
df_accepted = df_all.xs(' accepted',level='Time(s)/Cell Status',axis=1)
df_accepted.index = df_accepted.index+time_difference
df_z = df_accepted.apply(stats.zscore,axis=0)
plt.figure(figsize=[6,3])
plt.imshow(df_z.T,aspect='auto',vmin=0, vmax=5, cmap='Greys')
plt.colorbar()
plt.tight_layout()
plt.show()

## Load deconvoluted traces
r_path_ca_ED = [r for r in selected_result_ca if 'celltraces_ED.csv' in r]
print(r_path_ca_ED)

event_path = Ca2_folder + r_path_ca_ED[0]
df_event=pd.read_csv(event_path, header=0,skiprows=[1],index_col=0)
# Pivot the dataframe for all time stamps for easy handling
df_event_binary = df_event.pivot(columns=' Cell Name',values=' Value').fillna(0)
df_event_binary[df_event_binary>0] = 1 # Make deconvoluted values to 1
df_event_binary.index = df_event_binary.index+time_difference
df_event_binary_accepted = df_event_binary[df_accepted.columns]

# plt.plot(df_event_binary_accepted.to_numpy())
fix,ax = plt.subplots(figsize=[5,3])
Ca2_eventplot(df_event_binary_accepted,ax=ax)
# plt.xlim(xmin=0)
plt.tight_layout()
plt.show()

In [ ]:
#@title Load overview meta-data
overview_file_path = '/Users/friedrichjohenning/Dropbox/csv_F/mouse_overview.csv'
df_overview = pd.read_csv(overview_file_path,skiprows=2)
file_date = date
recording_date = '20'+file_date[0:2]+'/'+file_date[2:4]+'/'+file_date[4:6] # switch date to YYYY/MM/DD

select_overview = df_overview[df_overview['Mouse ID'] == mouse_id].copy()
select_overview = select_overview[select_overview['Recording date']==recording_date]
Pump1_solution = select_overview['Flavor 1'].to_numpy()[0].lower()
Pump2_solution = select_overview['Flavor 2'].to_numpy()[0].lower()
Food_deprivation = select_overview['Food Deprived'].to_numpy()[0].lower()
if 'CaMK2a' in select_overview['Genotype'].to_numpy()[0]:
  Genotype = 'CaMK2'
else:
  Genotype = select_overview['Genotype'].to_numpy()[0]
mouse_name = select_overview['Mouse Name'].to_numpy()[0]

if Pump1_solution != 'water':
    if Pump1_solution != 'sucrose':
        Pump1_solution = Pump1_solution + ' milk'
if Pump2_solution != 'water':
    if Pump2_solution != 'sucrose':
        Pump2_solution = Pump2_solution + ' milk'

print('Mouse id: '+ mouse_id)
print('Mouse Name: '+ mouse_name)
print('Genotype: ' + Genotype)
print('Date: '+str(file_date))
print('Food deprivation level: ' + Food_deprivation)
print('Pump 1: ' + Pump1_solution)
print('Pump 2: ' + Pump2_solution)

In [ ]:
#@title Sort cell order with binge level 
# set the number of bouts
x_shift = Pump_all_no_init[-1]
x_bar = np.array([x*180+300 for x in range(18)])
# x_bar = np.array(x_bar)
x_bar = x_bar[x_bar<x_shift]

delivery_per_bout = []
for x in x_bar:
    xmin = x
    xmax = x + 125 #+5 sec for fluctruations
    GPIO_filtered = filter_mask_small(filter_mask_large(Pump_all_no_init,xmax),xmin)
    if GPIO_filtered.any()>0:
      delivery_per_bout.append(len(GPIO_filtered))
    else:
      delivery_per_bout.append(0)

# find index of max deliveried bout
max_bout_id = np.argmax(delivery_per_bout)
baseline = 300
xmin = baseline + (max_bout_id)*180
xmax = xmin + 120+5 # +5 sec for fluctruations
GPIO_filtered = filter_mask_small(filter_mask_large(Pump_all_no_init,xmax),xmin)
xmin_sort = GPIO_filtered[1]
xmax_sort = GPIO_filtered[-1]

event_count = df_event_binary_accepted.loc[xmin_sort:xmax_sort,:].sum().values
isort = np.argsort(event_count)
sorted_cell_name = df_event_binary_accepted.columns[isort]
df_event_sorted = df_event_binary_accepted.reindex(columns=sorted_cell_name)

df_event_sorted.head()

In [ ]:
## separate the number of licks in 'binging mode' from number of licks in 'fast mode' for different exps
#@title Sort cell order with binge level 
# set the number of bouts



x_shift = Pump_all_no_init[-1]

x_bar = np.array([x*180+300 for x in range(18)])
# x_bar = np.array(x_bar)
x_bar = x_bar[x_bar<x_shift]

delivery_per_bout = []
slowdeliveries=0
fastdeliveries=0
for x in x_bar:
    xmin = x
    xmax = x + 125 #+5 sec for fluctruations
    GPIO_filtered = filter_mask_small(filter_mask_large(Pump_all_no_init,xmax),xmin)
    
    
    GPIO_filtereddiffs=np.ediff1d(GPIO_filtered)
    if GPIO_filtered.any()>0:
        
        for singleDiff in GPIO_filtereddiffs:
            if singleDiff <4:
                fastdeliveries=fastdeliveries+1
            else:
                slowdeliveries=slowdeliveries+1
    else:
        print('empty')
    
print (slowdeliveries)
print (fastdeliveries)
print (fastdeliveries/slowdeliveries)


In [ ]:
mouse_id = 'BES0233'
listeratios=[]
selected_result = [r for r in result if mouse_id in r]

fig, [ax0,ax1,ax2] = plt.subplots(figsize=[10,8],ncols=3,nrows=1,sharey=True,gridspec_kw={'width_ratios':[8,1,1]})
ax0.set_title(mouse_id)
pump_num = []
pump_time_max = []
dates = []

for idx,r in enumerate(tqdm(selected_result[:])):
  file_date = r.split('-')[-1].split('.')[0] + r.split('-')[-1].split('.')[1] + r.split('-')[-1].split('.')[2][:-1]
  dates.append(file_date)
  P1,P1C,P2,P2A,Pump_all = GPIO_event_calculate(Phenosys_folder + r)
  Pump1,Pump2 = select_pump(P1,P1C,P2,P2A)
  Pump1,Pump2,Pump_all_no_init = remove_init(Pump1,Pump2)
  pump_num.append(len(Pump_all_no_init))
  if len(Pump_all_no_init) == 0:
    pump_time_max.append(0)
  else:
    pump_time_max.append(Pump_all_no_init[-1])
  ax0.eventplot(Pump_all_no_init,linelengths = 0.8,linewidths=0.6,lineoffsets = idx,color=my_color_map[0])
  ax0.text(s=file_date,x=275,y=idx,va='center',ha='right')
  sns.despine(left=True)
  ax0.set_yticks([])
    
    
    
    
pump_time_max = np.max(pump_time_max)

x_shift = Pump_all_no_init[-1]

x_bar = np.array([x*180+300 for x in range(18)])
# x_bar = np.array(x_bar)
x_bar = x_bar[x_bar<x_shift]

delivery_per_bout = []

for idx,r in enumerate(tqdm(selected_result[:])):
    file_date = r.split('-')[-1].split('.')[0] + r.split('-')[-1].split('.')[1] + r.split('-')[-1].split('.')[2][:-1]
    dates.append(file_date)
    P1,P1C,P2,P2A,Pump_all = GPIO_event_calculate(Phenosys_folder + r)
    Pump1,Pump2 = select_pump(P1,P1C,P2,P2A)
    Pump1,Pump2,Pump_all_no_init = remove_init(Pump1,Pump2)
    x_shift = Pump_all_no_init[-1]

    x_bar = np.array([x*180+300 for x in range(18)])
    # x_bar = np.array(x_bar)
    x_bar = x_bar[x_bar<x_shift]
    
    
    slowdeliveries=0
    fastdeliveries=0
    for x in x_bar:
        xmin = x
        xmax = x + 125 #+5 sec for fluctruations
        GPIO_filtered = filter_mask_small(filter_mask_large(Pump_all_no_init,xmax),xmin)
    
        
        GPIO_filtereddiffs=np.ediff1d(GPIO_filtered)
        print (x)
        print (GPIO_filtered.size)
        print (GPIO_filtereddiffs.size)
        if GPIO_filtereddiffs.size==0:
            GPIO_filtereddiffs=[4.2]
        for singleDiff in GPIO_filtereddiffs:
            if singleDiff <4:
                fastdeliveries=fastdeliveries+1
            else:
                slowdeliveries=slowdeliveries+1
    print (file_date)
    print (slowdeliveries)
    print (fastdeliveries)
    verhaeltnis=fastdeliveries/(slowdeliveries+fastdeliveries)
    print (verhaeltnis)
    listeratios.append(verhaeltnis)
print (listeratios)
listeratios=np.array(listeratios)

xsteps = altspace(250,250,20)
xsteps = xsteps[xsteps<np.max(pump_time_max+100)] 
ax0.set_xticks(xsteps)
ax0.set_xlabel('Seconds')
ax0.set_ylim(ymin=-1)

ax1.barh(range(len(pump_num)),width=pump_num/np.max(pump_num)*100,left=0,color=my_color_map[1])
for idx,num in enumerate(pump_num):
  ax1.text(s=num,x=-5,y=idx,va='center',ha='right')
  ax1.text(s=str(int(round(num/np.max(pump_num)*100)))+'%',x=160,y=idx,va='center',ha='right')
ax1.set_xlabel('Total deliveries (%)')
plt.tight_layout()

ax2.barh(range(len(listeratios)),width=listeratios*100,left=0,color=my_color_map[2])
for idx,num in enumerate(listeratios):
  #ax2.text(s=num,x=-5,y=idx,va='center',ha='right')
  listeratiosr=np.around(listeratios*100)
  ax2.text(s=str(listeratiosr[idx])+'%',x=160,y=idx,va='center',ha='right')
ax2.set_xlabel('fraction fast deliveries (%)')
plt.tight_layout()

print (dates)
if save_fig:
  plt.savefig('/content/drive/My Drive/inscopix_csv/Fig_output/'+mouse_id+'_behavior_plot.pdf', pdi=150)


In [ ]:
print (x_bar) 
print (x_shift)
print (xmax)
print (Pump_all_no_init[-1])
print (pump_num)

In [ ]:
print(dates)

In [ ]:
#@title Plots for global events (sorted)

# Strings setups
title, plot_name = title_plot_name(mouse_id, Food_deprivation, file_date)

# Initiate plotting
fig, [ax0,ax1] = plt.subplots(nrows=2,ncols=1,figsize=[10,6], gridspec_kw={'height_ratios': [1, 7],'hspace':0},sharex=True)

## Number of pump1 and pump2 for the delivery n on the fig legend
deliver_milk, deliver_water = length_delivery(Pump1_clean,Pump2_clean)
y_offset = df_event_binary.shape[-1]+3

## Plotting
ax0 = GPIO_eventplot(Pump1_clean,Pump2_clean, ax0) 
ax1 = Ca2_eventplot(df_event_sorted, ax1)

# Plot which bout is selected for sorting
ax0.hlines(xmin=xmin_sort,xmax=xmax_sort,y=1.75,linewidth=1,color=my_color_map[1])

## Aesthetic & lables
legend_hidden_line_red = mlines.Line2D([], [], color=my_color_map[2], markersize=6, marker='|',label=Pump1_solution + ' n='+str(deliver_milk),linestyle='')
legend_hidden_line_green = mlines.Line2D([], [], color=my_color_map[1], markersize=6, marker='|',label=Pump2_solution + ' n='+str(deliver_water),linestyle='')
ax0.legend(handles=[legend_hidden_line_red, legend_hidden_line_green],fontsize=8,loc=2,frameon=False)

sns.despine()
sns.despine(ax=ax0, left=True,bottom=True)
ax0.set_yticks([])

ax1.set_xlabel('Second'); ax1.set_ylabel('# Neuron')
ax1.set_yticks([1,df_event_binary_accepted.shape[-1]])
plt.xlim(xmin=0); plt.ylim(ymin=0,ymax=df_event_binary_accepted.shape[-1]+2)
ax0.set_title(title)
plt.tight_layout()
if save_fig:
  plt.savefig('/content/drive/My Drive/inscopix_csv/Fig_output/'+mouse_id+str(date)+'_populational_plot.pdf', pdi=150)

In [ ]:
#@title Checking lick sensor data after each deliveries
## Checking for lick sensor data for whether mouse really drank someting or just error touches.
# First to separate slow eating & binge eating

## Automatic identify binge bouts from delivery numbers and inter-event-intervals.
# set the number of bouts
x_shift = Pump_all_no_init[-1]
x_bar = np.array([x*180+300 for x in range(18)])
x_bar = x_bar[x_bar<x_shift]

delivery_type = np.zeros(len(x_bar))

## Set all slow eating delivery rounds into type 0, binge rounds into type 1

for idx, x_ in enumerate(x_bar):
  num_delivery = len(filter_mask_range(Pump_all_no_init,x_,x_+120)) # check Pump_all_no_init since it contains both pump deliveries
  # print(num_delivery)
  if num_delivery<27:
    if len(filter_mask_range(Pump_all_no_init,x_,x_+120))>1:
      if np.diff(filter_mask_range(Pump_all_no_init,x_,x_+120)).min()>4:
      # print(idx,np.diff(filter_mask_range(Pump_all_no_init,x_,x_+120)).min())
        delivery_type[idx] = 0
      else:
        delivery_type[idx] = 1
    else:
      delivery_type[idx] = 0 # length is <=1 so there is only 1 or 0 delivery
  else:
    delivery_type[idx] = 1
print(delivery_type)
slow_deliveries_milk = np.array([])
slow_deliveries_water = np.array([])
for idx in range(len(delivery_type)):
  if delivery_type[idx] == 0:
    slow_deliveries_milk  = np.concatenate([slow_deliveries_milk,filter_mask_range(Pump1_clean,x_bar[idx],x_bar[idx]+120)])
    slow_deliveries_water = np.concatenate([slow_deliveries_water,filter_mask_range(Pump2_clean,x_bar[idx],x_bar[idx]+120)])

## Now checking lick sensor data 
# Make sure there is at least 1 lick after the pump is activated.
L1,L2,Lick_all = lick_event_calculate(Phenosys_folder + r_path_pheno[0])

# It's very unlikely that pump activation is done due to the triggering from another lick sensor, I assume the activation is always related to the same lick spout/sensor
new_idx = []
licks_pump1 = []
lick_timesatmp1 = []
lick_timesatmp1_align = []
lick_timesatmp1_align_cell = {}

plt.figure(figsize=[6,5])
for idx, t in enumerate(slow_deliveries_milk):
  plt.eventplot(filter_mask_range(Lick_all,t,t+4)-t,lineoffsets=idx)
  if len(filter_mask_range(Lick_all,t,t+4)) ==0:
    plt.axhspan(ymin=idx-0.5,ymax=idx+0.5,alpha=0.1,color='C0')
  plt.text(x=4.15,y=idx,s=len(filter_mask_range(Lick_all,t,t+4)),ha='right',va='center', fontsize=7)
  if filter_mask_range(Lick_all,t,t+4).any() >0:
    new_idx.append(idx)
    licks_pump1.append(len(filter_mask_range(Lick_all,t,t+4)))
    lick_timesatmp1.append(filter_mask_range(Lick_all,t,t+4).min())
    lick_timesatmp1_align.append(filter_mask_range(Lick_all,t,t+4).min()-t)
    lick_timesatmp1_align_cell[idx] = lick_timesatmp1_align[-1]
plt.axvline(x=0,ymin=-0.5,ymax=idx+0.5,ls=':',color='k')
plt.ylim(ymin=-0.5,ymax=idx+0.5)
plt.title('Licks after {} deliveries (n={})'.format(Pump1_solution, idx+1))
sns.despine()
plt.tight_layout()
plt.show()
slow_deliveries_milk = slow_deliveries_milk[new_idx]

new_idx = []
licks_pump2 = []
lick_timesatmp2 = []
lick_timesatmp2_align = []

plt.figure(figsize=[6,5])
for idx, t in enumerate(slow_deliveries_water):
  plt.eventplot(filter_mask_range(Lick_all,t,t+4)-t,lineoffsets=idx)
  if len(filter_mask_range(Lick_all,t,t+4)) ==0:
    plt.axhspan(ymin=idx-0.5,ymax=idx+0.5,alpha=0.1,color='C1')
  plt.text(x=4.15,y=idx,s=len(filter_mask_range(Lick_all,t,t+4)),ha='right',va='center', fontsize=7)
  if filter_mask_range(Lick_all,t,t+4).any() >0:
    new_idx.append(idx)
    licks_pump2.append(len(filter_mask_range(Lick_all,t,t+4)))
    lick_timesatmp2.append(filter_mask_range(Lick_all,t,t+4).min())
    lick_timesatmp2_align.append(filter_mask_range(Lick_all,t,t+4).min()-t)

plt.axvline(x=0,ymin=-0.5,ymax=idx+0.5,ls=':',color='k')
plt.ylim(ymin=-0.5,ymax=idx+0.5)
plt.title('Licks after {} deliveries (n={})'.format(Pump2_solution, idx+1))
sns.despine()
plt.tight_layout()
plt.show()

slow_deliveries_water = slow_deliveries_water[new_idx]

print('Pump1: \n {} \n {} deliveries'.format(Pump1_solution, len(slow_deliveries_milk)))
print()
print('Pump2: \n {} \n {} deliveries'.format(Pump2_solution, len(slow_deliveries_water)))


In [ ]:
## Comparison of licks after milk/water deliveries

licks_df_tidy = pd.DataFrame(
                  {Pump1_solution: pd.Series(licks_pump1),
                   Pump2_solution: pd.Series(licks_pump2)
                  }).melt().dropna()

fig, ax = plt.subplots(figsize=[3,3])
sns.boxplot(data=licks_df_tidy,x='variable',y='value',width=0.5)
# sns.violinplot(data=licks_df_tidy,x='variable',y='value',width=0.5)
sns.swarmplot(data=licks_df_tidy,x='variable',y='value',color='k',alpha=0.6)
sns.despine()
ax.set(xlabel=None,ylabel='Licks')
ax.yaxis.get_major_locator().set_params(integer=True) # set y ticks to only integers
plt.tight_layout()

In [ ]:
## Think about to make the comparison of pump activation alignment and first lick after pump activation.
## Generate PSTH for each cells with milk and water trials
## Consider to drop the cell name (C0, C1...) here so we don't deal with naming but just int index

plot_viz = True

# Use dict for storaging data from each cell
PSTH_trace_milk = {} 
PSTH_trace_water = {} 
PSTH_trace_milk_lick = {} 
PSTH_trace_water_lick = {} 

z_score = True

if z_score:
  data = df_z.copy()
else:
  data = df_accepted.copy()

cellset = data.columns

num_bins = 100
t_scale = np.linspace(-1,4,100)

# np.random.seed(0)

## generate neural matrix in dict
for cell in cellset:
  PSTH_trace = np.zeros([len(slow_deliveries_milk),num_bins])
  for trial, time in enumerate(slow_deliveries_milk):
    PSTH_trace[trial,:] = data[cell][time-1.5:time+4].reset_index(drop=True)[0:num_bins]
  PSTH_trace_milk[cell] = PSTH_trace
  PSTH_trace = np.zeros([len(slow_deliveries_water),num_bins])
  for trial, time in enumerate(slow_deliveries_water):
    PSTH_trace[trial,:] = data[cell][time-1.5:time+4].reset_index(drop=True)[0:num_bins]
  PSTH_trace_water[cell] = PSTH_trace

## generate neural matrix in dict
for cell in cellset:
  PSTH_trace = np.zeros([len(slow_deliveries_milk),num_bins])
  for trial, time in enumerate(lick_timesatmp1):
    PSTH_trace[trial,:] = data[cell][time-1.5:time+4].reset_index(drop=True)[0:num_bins]
  PSTH_trace_milk_lick[cell] = PSTH_trace
  PSTH_trace = np.zeros([len(slow_deliveries_water),num_bins])
  for trial, time in enumerate(lick_timesatmp2):
    PSTH_trace[trial,:] = data[cell][time-1.5:time+4].reset_index(drop=True)[0:num_bins]
  PSTH_trace_water_lick[cell] = PSTH_trace

for _ in range(5):
  random_index = np.random.choice(cellset,size=1,replace=False)
  fig, axes = plt.subplots(figsize=[10,3],sharex=True,nrows=1,ncols=2)
  for col in range(2):
    idx_ = 0
    if col ==0:
      axes[col].plot(t_scale,PSTH_trace_milk[random_index[idx_]].mean(axis=0),label='milk n={}'.format(len(slow_deliveries_milk)))
      axes[col].plot(t_scale,PSTH_trace_water[random_index[idx_]].mean(axis=0),label='water n={}'.format(len(slow_deliveries_water)))
      # pd.DataFrame.from_dict(PSTH_trace_milk)
      # plt.imshow(PSTH_trace_milk[random_index].mean(axis=0).T,aspect='auto',vmin=0, vmax=5, cmap='Greys')
    else:
      axes[col].plot(t_scale,PSTH_trace_milk_lick[random_index[idx_]].mean(axis=0),label='milk n={}'.format(len(slow_deliveries_milk)))
      axes[col].plot(t_scale,PSTH_trace_water_lick[random_index[idx_]].mean(axis=0),label='water n={}'.format(len(slow_deliveries_water)))
      # plt.imshow(PSTH_trace_milk_lick.T,aspect='auto',vmin=0, vmax=5, cmap='Greys')
    axes[col].axvspan(xmin=0,xmax=4,alpha=0.05,color='C3')
    axes[col].set_title(random_index[idx_]+' PSTH (trial-averaged)')
    if z_score:
      axes[col].set_ylabel('∆F/F (z-scored)')
    else:
      axes[col].set_ylabel('∆F/F (A.U.)')
    if col ==0:
      axes[col].set_xlabel('Time from delivery (seconds)')
    else:
      axes[col].set_xlabel('Time from first lick (seconds)')
    axes[col].legend()
    sns.despine()
  plt.tight_layout()

# if plot_viz:
#   for i in range(1):
#     random_index = np.random.choice(cellset,size=6,replace=False)
#     fig, axes = plt.subplots(figsize=[12,6],sharex=True,nrows=2,ncols=3)
#     for row in range(2):
#       for col in range(3):
#         idx_ = row + col 
#         axes[row,col].plot(t_scale,PSTH_trace_milk[random_index[idx_]].mean(axis=0),label='milk n={}'.format(len(slow_deliveries_milk)))
#         axes[row,col].plot(t_scale,PSTH_trace_water[random_index[idx_]].mean(axis=0),label='water n={}'.format(len(slow_deliveries_water)))
#         axes[row,col].axvspan(xmin=0,xmax=4,alpha=0.05,color='C3')
#         axes[row,col].set_title(random_index[idx_]+' PSTH (trial-averaged)')
#         if col == 0:
#           if z_score:
#             axes[row,col].set_ylabel('∆F/F (z-scored)')
#           else:
#             axes[row,col].set_ylabel('∆F/F (A.U.)')
#         if row == 1:
#           axes[row,col].set_xlabel('Time from delivery (seconds)')
#         axes[row,col].legend()
#         sns.despine()
#     plt.tight_layout()

# # plt.imshow()